# LOC Anomaly Detection — Tiered Pipeline

Reads pre-aggregated rate tables built by `loc_agg.sql`, then runs a two-tier
detection pipeline for each population (M&R FFS, C&S DSNP, OAH):

1. **Tier 1 — Percentile flagging**: flags entities in the top/bottom 10th percentile of each rate feature within their dimension. Directly actionable, no model needed.
2. **Tier 2 — Isolation Forest**: catches entities with an unusual *combination* of rates that no single KPI flag would surface.

**Run order:**
1. Run `loc_agg.sql` in Snowflake (creates the 3 `kn_loc_*_agg` tables)
2. Run this notebook (reads those tables, tiers 1 & 2, exports CSVs)

## Setup & Configuration

In [1]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine

warnings.filterwarnings("ignore", category = RuntimeWarning)

In [2]:
# --- update each run (must match loc_agg.sql) ---
RUN_DATE = "202604"

OUTPUT_DIR = r"C:\Users\knguy139\Documents\Projects\Data\Output"
os.makedirs(OUTPUT_DIR, exist_ok = True)

# one entry per population: (snowflake table, file prefix)
POPULATIONS = [
    (f"tmp_1m.kn_loc_mnr_agg_{RUN_DATE}", "mnr"),
    (f"tmp_1m.kn_loc_cns_agg_{RUN_DATE}", "cns"),
    (f"tmp_1m.kn_loc_oah_agg_{RUN_DATE}", "oah"),
]

# tier 1 thresholds
TOP_PCT    = 0.10
BOTTOM_PCT = 0.10

# tier 2 settings
CONTAMINATION = 0.05   # business decision, not statistical
TOP_N         = 20     # narratives per dimension

# rate features (column names from loc_agg.sql)
RATE_FEATURES = [
    "adr_rate",
    "persistent_adr_rate",
    "persistency",
    "md_review_rate",
    "appeal_rate",
    "appeal_overturn_rate",
    "p2p_rate",
    "p2p_overturn_rate",
    "mcr_reconsideration_rate",
    "mcr_overturn_rate",
    "member_appeal_rate",
    "member_appeal_overturn_rate",
    "pre_auth_rate",
    "auth_per_k",
]

# labels and formats for narratives
FEATURE_META = {
    "adr_rate":                    ("ADR rate",                                "{:.1%}"),
    "persistent_adr_rate":         ("Persistent ADR rate",                     "{:.1%}"),
    "persistency":                 ("Persistency (persistent / initial ADR)",  "{:.1%}"),
    "md_review_rate":              ("MD review rate",                          "{:.1%}"),
    "appeal_rate":                 ("Appeal rate (% of ADRs)",                 "{:.1%}"),
    "appeal_overturn_rate":        ("Appeal overturn rate (% of ADRs)",        "{:.1%}"),
    "p2p_rate":                    ("P2P rate (% of ADRs)",                    "{:.1%}"),
    "p2p_overturn_rate":           ("P2P overturn rate (% of ADRs)",           "{:.1%}"),
    "mcr_reconsideration_rate":    ("MCR reconsideration rate (% of ADRs)",    "{:.1%}"),
    "mcr_overturn_rate":           ("MCR overturn rate (% of ADRs)",           "{:.1%}"),
    "member_appeal_rate":          ("Member appeal rate (% of ADRs)",          "{:.1%}"),
    "member_appeal_overturn_rate": ("Member appeal overturn rate (% of ADRs)", "{:.1%}"),
    "pre_auth_rate":               ("Pre-auth rate",                           "{:.1%}"),
    "auth_per_k":                  ("Auth per 1,000 members",                  "{:.1f}"),
}

## Connect to Snowflake

In [3]:
engine = create_engine(
    URL(
        account = "UHG-UHGDWAAS",
        user = "KHANG.NGUYEN@UHC.COM",
        authenticator = "externalbrowser",
        role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
        warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
        database = "VING_PRD_TREND_DB",
        schema = "TMP_1M"
    )
)

## Load data

Pull all 3 population tables from Snowflake into a dict keyed by prefix.

In [4]:
data = {}
for table, prefix in POPULATIONS:
    with engine.connect() as conn:
        data[prefix] = (
            pd.read_sql(f"select * from {table}", conn)
            .rename(columns = str.lower)
        )
    print(f"{prefix.upper()}: {len(data[prefix]):,} rows")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVLRbpswFP0V5D2DDYUssZJUrGm0SNmGErpJe5mMMcSNsZltQrqvnyGJ1D200h4sWfY595x7z53fnxvhnZg2XMkFCAMEPCapKrmsF%2BApX%2FtT4BlLZEmEkmwBXpgB98u5IY1ocdrZg9yx3x0z1nOFpMHDxwJ0WmJFDDdYkoYZbCnep1%2B2OAoQJsYwbZ0cuFJKw53WwdoWQ9j3fdDfBUrXMEIIQTSDDjVAPoBXEu37Gq1WVlElbpSz6%2BkNiRCieJBwCKeQXYmfuLyM4D2V4gIy%2BHOeZ372bZ8DL71196Ck6Rqm90yfOGVPu%2B3FgHEOukPtu1P2hJhfBxIYqfpKkCOjqmk762oG7gYrVkKhau4mtVktQHvk5Wk1zcTjKjV1%2FCMvdJcfp%2FwFZYdHsl1nf9JdweokWT8fn8%2BUAu%2F7LddoyHVjTMc2ckjTuicUTXwU%2B1GchzFOIozugkkU%2FQTeyqXJJbEj82Z59BE0nGplVGWVFFyy0WVZoKQilPh0GhE%2FLmalX8xo4qNqEheTj0kSRyEcMovAZW%2FwaEQv%2F28ac%2Fiae13Ary6TzSpTgtMXb610Q%2BzbkYVBOL7w0q9GKGYN4SItS82McdEJofoHzYh1e251xwBcXlT%2F3fTlXw%3D%3D&RelayState=ver%3A3-hint%3A260091978973398-ETMsDgAAAZ2%2F%2BfimABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEBwQbNbcCMTaKiB%2BnJxf2PQAAACQ6B

DatabaseError: Execution failed on sql 'select * from tmp_1m.kn_loc_mnr_agg_202604': (snowflake.connector.errors.ProgrammingError) 002003 (42S02): SQL compilation error:
Object 'VING_PRD_TREND_DB.TMP_1M.KN_LOC_MNR_AGG_202604' does not exist or not authorized.
[SQL: select * from tmp_1m.kn_loc_mnr_agg_202604]
(Background on this error at: https://sqlalche.me/e/20/f405)

## Quick look at the data

Sanity check: dimensions present, row counts, rate feature coverage.

In [ ]:
for prefix, df in data.items():
    dims = df["_dimension"].value_counts()
    rate_present = [f for f in RATE_FEATURES if f in df.columns]
    print(f"--- {prefix.upper()} ---")
    print(f"  columns: {len(df.columns)}  |  rate features found: {len(rate_present)}/{len(RATE_FEATURES)}")
    print(f"  dimensions:")
    for dim, n in dims.items():
        print(f"    {dim}: {n:,}")
    print()

## Tier 1 — Percentile flagging

For each dimension x rate feature, rank every entity against its peers.  
Flag anyone in the top or bottom 10th percentile.  
This is the primary, directly-actionable output.

In [ ]:
pctl_results = {}

for prefix, df in data.items():
    pctl_rows = []
    for dim, group in df.groupby("_dimension"):
        for feat in RATE_FEATURES:
            if feat not in group.columns:
                continue
            ranked = group[["_dim_value", "case_count", feat]].dropna(subset = [feat]).copy()
            if ranked.empty:
                continue
            ranked["percentile"]  = ranked[feat].rank(pct = True)
            ranked["peer_median"] = ranked[feat].median()
            ranked["flag_high"]   = ranked["percentile"] >= (1 - TOP_PCT)
            ranked["flag_low"]    = ranked["percentile"] <= BOTTOM_PCT
            ranked["_dimension"]  = dim
            ranked["_feature"]    = feat
            pctl_rows.append(ranked)

    pctl = pd.concat(pctl_rows, ignore_index = True) if pctl_rows else pd.DataFrame()
    pctl_results[prefix] = pctl

    n_flagged  = pctl["flag_high"].sum() + pctl["flag_low"].sum()
    n_entities = pctl.loc[pctl["flag_high"] | pctl["flag_low"], "_dim_value"].nunique()
    print(f"{prefix.upper()}: {n_flagged:,} flags across {n_entities:,} entities")

## Tier 2 — Isolation Forest

Secondary sweep: fit an Isolation Forest per dimension to catch entities  
whose *combination* of rates is unusual, even if no single rate is extreme.  
Features are scaled so no single rate dominates.

In [ ]:
if_results = {}

for prefix, df in data.items():
    all_scored = []
    for dim, group in df.groupby("_dimension"):
        available = [f for f in RATE_FEATURES if f in group.columns and group[f].notna().any()]
        if not available:
            all_scored.append(group)
            continue

        X = group[available].fillna(group[available].median())
        X_scaled = StandardScaler().fit_transform(X)

        model = IsolationForest(
            n_estimators  = 200,
            contamination = CONTAMINATION,
            random_state  = 42,
            n_jobs        = -1,
        )
        model.fit(X_scaled)

        scores = model.decision_function(X_scaled)
        scored = group.assign(
            raw_score      = scores,
            _features_used = ",".join(available),
        )
        all_scored.append(scored)

    if_scored = (
        pd.concat(all_scored, ignore_index = True)
        .sort_values(["_dimension", "raw_score"], ascending = [True, True])
    )
    if_results[prefix] = if_scored

    n_flagged = (if_scored["raw_score"] < 0).sum() if "raw_score" in if_scored.columns else 0
    print(f"{prefix.upper()}: {n_flagged:,} entities flagged by IF")

## Z-scores & top reasons

Isolation Forest is a black box. Z-scores add interpretability after the fact:  
for each flagged entity, which features are most extreme relative to the  
dimension mean? We tag the top 3.

In [ ]:
def top_reasons(row):
    """Top 3 most extreme z-score features for one entity row."""
    z_vals = {
        feat: abs(row[f"{feat}_z"])
        for feat in RATE_FEATURES
        if f"{feat}_z" in row.index and pd.notna(row[f"{feat}_z"])
    }
    top3 = sorted(z_vals, key = z_vals.get, reverse = True)[:3]

    reasons = {}
    for i, feat in enumerate(top3, start = 1):
        z         = row[f"{feat}_z"]
        direction = "above" if z > 0 else "below"
        reasons[f"top_reason_{i}"] = f"{feat} ({z:+.1f}\u03c3 {direction} peer mean)"
    for i in range(len(top3) + 1, 4):
        reasons[f"top_reason_{i}"] = ""

    return pd.Series(reasons)


for prefix in if_results:
    scored = if_results[prefix]

    # compute z-scores within each dimension
    enriched_parts = []
    for dim, group in scored.groupby("_dimension"):
        z_cols = {
            f"{feat}_z": (group[feat] - group[feat].mean()) / group[feat].std()
            for feat in RATE_FEATURES
            if feat in group.columns and group[feat].std() > 0
        }
        enriched = group.assign(**z_cols)
        enriched[["top_reason_1", "top_reason_2", "top_reason_3"]] = enriched.apply(
            top_reasons, axis = 1
        )
        enriched_parts.append(enriched)

    if_results[prefix] = pd.concat(enriched_parts, ignore_index = True)

print("Z-scores and top reasons added.")

## Narratives

For the top 20 most anomalous entities per dimension, generate a  
human-readable paragraph explaining what's unusual about their pattern.

In [ ]:
def generate_narrative(row, peer_medians, dim):
    """Plain-English paragraph explaining why one entity is anomalous."""
    lines = [
        f"{dim} = {row['_dim_value']}  |  "
        f"cases = {int(row['case_count']):,}  |  "
        f"anomaly score = {row['raw_score']:.3f}"
    ]

    # only surface features that are notably extreme (>1.5 sigma)
    for feat, (label, fmt) in FEATURE_META.items():
        z_col = f"{feat}_z"
        if feat not in row.index or pd.isna(row.get(feat)):
            continue
        z = row.get(z_col, np.nan)
        if pd.isna(z) or abs(z) < 1.5:
            continue
        direction = "above" if z > 0 else "below"
        lines.append(
            f"  \u2022 {label}: {fmt.format(row[feat])} "
            f"vs. peer median {fmt.format(peer_medians.get(feat, np.nan))} "
            f"({z:+.1f}\u03c3 {direction})"
        )

    if len(lines) == 1:
        lines.append("  \u2022 No individual feature stands out; flagged on combined pattern.")

    return "\n".join(lines)


narrative_results = {}

for prefix, if_scored in if_results.items():
    narrative_rows = []
    for dim, group in if_scored.groupby("_dimension"):
        top_n        = group.nsmallest(TOP_N, "raw_score")
        peer_medians = group[[f for f in RATE_FEATURES if f in group.columns]].median()
        for _, row in top_n.iterrows():
            narrative_rows.append({
                "dimension":  dim,
                "dim_value":  row["_dim_value"],
                "case_count": int(row["case_count"]),
                "raw_score":  row["raw_score"],
                "narrative":  generate_narrative(row, peer_medians, dim),
            })
    narrative_results[prefix] = pd.DataFrame(narrative_rows)

    print(f"{prefix.upper()}: {len(narrative_rows):,} narratives generated")

## Combine tier 1 + tier 2 flags

Merge both tiers into a single entity-level table with a `tier` column:  
`"percentile"`, `"isolation_forest"`, or `"both"`.

In [ ]:
combined_results = {}

for prefix in data:
    pctl      = pctl_results[prefix]
    if_scored = if_results[prefix]

    # tier 1: entities flagged on any feature
    t1_flagged = set()
    if not pctl.empty:
        flagged_rows = pctl[pctl["flag_high"] | pctl["flag_low"]]
        t1_flagged = set(
            flagged_rows[["_dimension", "_dim_value"]]
            .apply(tuple, axis = 1)
        )

    # tier 2: entities with negative raw_score
    t2_flagged = set()
    if "raw_score" in if_scored.columns:
        neg = if_scored[if_scored["raw_score"] < 0]
        t2_flagged = set(
            neg[["_dimension", "_dim_value"]]
            .apply(tuple, axis = 1)
        )

    def assign_tier(row):
        key = (row["_dimension"], row["_dim_value"])
        in_t1 = key in t1_flagged
        in_t2 = key in t2_flagged
        if in_t1 and in_t2:
            return "both"
        elif in_t1:
            return "percentile"
        elif in_t2:
            return "isolation_forest"
        return ""

    combined = if_scored.assign(tier = if_scored.apply(assign_tier, axis = 1))
    combined_results[prefix] = combined

    flagged_either = (combined["tier"] != "").sum()
    print(f"{prefix.upper()}: {flagged_either:,} entities flagged by at least one tier")

## Export CSVs

In [ ]:
id_cols    = ["_dimension", "_dim_value", "case_count", "membership"]
rate_cols  = [f for f in RATE_FEATURES]  # all expected
score_cols = ["raw_score", "_features_used", "top_reason_1", "top_reason_2", "top_reason_3"]

for prefix in data:
    pctl     = pctl_results[prefix]
    if_scored = if_results[prefix]
    combined = combined_results[prefix]
    narr     = narrative_results[prefix]

    # tier 1 percentile flags
    pctl_csv = os.path.join(OUTPUT_DIR, f"loc_pctl_{prefix}_{RUN_DATE}.csv")
    if not pctl.empty:
        pctl.to_csv(pctl_csv, index = False)

    # tier 2 scores
    if_csv = os.path.join(OUTPUT_DIR, f"loc_if_{prefix}_{RUN_DATE}.csv")
    if_scored[[c for c in id_cols + rate_cols + score_cols if c in if_scored.columns]].to_csv(
        if_csv, index = False
    )

    # narratives
    narrative_csv = os.path.join(OUTPUT_DIR, f"loc_narratives_{prefix}_{RUN_DATE}.csv")
    narr.to_csv(narrative_csv, index = False)

    # combined
    combined_csv = os.path.join(OUTPUT_DIR, f"loc_combined_{prefix}_{RUN_DATE}.csv")
    combined[[c for c in id_cols + rate_cols + ["raw_score", "tier"] if c in combined.columns]].to_csv(
        combined_csv, index = False
    )

    print(f"{prefix.upper()}: 4 CSVs written to {OUTPUT_DIR}")

## Summary

In [ ]:
for prefix in data:
    pctl     = pctl_results[prefix]
    combined = combined_results[prefix]

    print(f"--- {prefix.upper()} ---")
    t1_flags   = pctl["flag_high"].sum() + pctl["flag_low"].sum()
    t2_flagged = (combined["raw_score"] < 0).sum() if "raw_score" in combined.columns else 0
    print(f"  Tier 1 (percentile):  {t1_flags:,} feature-level flags")
    print(f"  Tier 2 (IF):          {t2_flagged:,} entities flagged")

    for tier_val in ["percentile", "isolation_forest", "both"]:
        n = (combined["tier"] == tier_val).sum()
        if n > 0:
            print(f"    tier={tier_val}: {n:,}")
    print()

In [ ]:
engine.dispose()